# Absolute colour–magnitude diagram of Gaia DR4 microlensing sources

Use notebook 02's joined CSV to plot Paczyński-0 baseline source colours and
absolute G magnitudes derived from Paczyński-3 source parallax. Apply explicit
fit-level and catalogue-quality cuts, with optional extinction corrections.
Save PNG/PDF figures, the selected-source CSV and plot provenance.

The density background is downloaded automatically from `gaiadr4.gaia_source`
on first use and cached locally. It is a reproducible random all-sky sample of
stars passing the stated CMD quality cuts. Set `USE_REFERENCE_BACKGROUND=False`
for a source-only plot.

The data are described in the Gaia DR4 publication by **Wyrzykowski et al. (2026)**.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from io import BytesIO
from getpass import getpass
import gzip, hashlib, json, re, socket, subprocess, time
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from matplotlib.colors import LogNorm


def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "01_archive_access.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the gaia-dr4-microlensing-notebooks folder.")

ROOT = find_project_root(Path.cwd())

INPUT_PATH = ROOT / 'outputs' / 'gaiadr4' / 'gaia_dr4_microlensing_events_joined.csv'
REFERENCE_CACHE = ROOT / 'data' / 'dr4_cmd_reference'
ARCHIVE_URL = 'https://gea.esac.esa.int/'
TAP_SCHEMA = 'gaiadr4'
DATA_RELEASE = 'Gaia DR4'
AUTHENTICATE = False
GAIA_USERNAME = 'YOUR_GAIA_USERNAME'
AUTH_METHOD = 'prompt'  # Optional: keychain, using your own macOS entry
KEYCHAIN_SERVICE = 'gaia-archive'
SMOKE_SOURCE_ID = '704967037090946688'
JOB_WAIT_SECONDS = 1800
NETWORK_TIMEOUT_SECONDS = 120
OUTPUT_STEM = ROOT / 'outputs' / 'gaiadr4' / 'microlensing_sources_absolute_cmd_quality_cut'
DEREDDEN = True
USE_REFERENCE_BACKGROUND = True
REFERENCE_RANDOM_INDEX_LIMIT = 2_000_000  # Random-index window, not a sky/magnitude cut
REFERENCE_MAX_STARS = 150_000  # Cap after finite/quality cuts; use fewer if fewer qualify
REFERENCE_SEED = 2026  # Reproducible local subsampling when more than the cap qualify
REFERENCE_MIN_PARALLAX_SNR = 5.0

# Catalogue cuts: conservative Gaia/all_source photometric and astrometric reliability.
MAX_RUWE = 1.4
MIN_VISIBILITY_PERIODS = 9
MIN_G_FLUX_SNR = 50
MIN_BP_RP_FLUX_SNR = 20

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Missing pre-downloaded joined table: {INPUT_PATH}')
OUTPUT_STEM.parent.mkdir(parents=True, exist_ok=True)
print(INPUT_PATH)
print(f'Dereddening: {DEREDDEN}')

In [ ]:
events = Table.read(INPUT_PATH, format='ascii.csv')
print(f'Loaded {len(events):,} joined microlensing rows with {len(events.colnames):,} columns.')

def numeric(column):
    """Return an astropy column as floats, converting masked values to NaN."""
    return np.asarray(np.ma.asarray(column, dtype=float).filled(np.nan), dtype=float)

def boolean(column):
    """Read Gaia boolean columns while treating an absent/masked value as False."""
    value = np.ma.asarray(column).filled(False)
    if value.dtype.kind in 'b':
        return np.asarray(value, dtype=bool)
    return np.char.lower(np.asarray(value, dtype=str)) == 'true'

# Paczynski-0 baseline source photometry and Paczynski-3 source parallax.
g0 = numeric(events['paczynski0_g0'])
g0_error = numeric(events['paczynski0_g0_error'])
bp0 = numeric(events['paczynski0_bp0'])
bp0_error = numeric(events['paczynski0_bp0_error'])
rp0 = numeric(events['paczynski0_rp0'])
rp0_error = numeric(events['paczynski0_rp0_error'])
source_parallax = numeric(events['paczynski3_varpi_source'])
source_parallax_error = numeric(events['paczynski3_varpi_source_error'])

colour = bp0 - rp0
colour_error = np.hypot(bp0_error, rp0_error)
source_parallax_snr = np.divide(
    source_parallax, source_parallax_error,
    out=np.full(len(events), np.nan), where=source_parallax_error > 0,
)

# Gaia/all_source columns available in the pre-downloaded joined table.
ruwe = numeric(events['ruwe'])
visibility_periods = numeric(events['visibility_periods_used'])
g_flux_snr = numeric(events['phot_g_mean_flux_over_error'])
bp_flux_snr = numeric(events['phot_bp_mean_flux_over_error'])
rp_flux_snr = numeric(events['phot_rp_mean_flux_over_error'])
unreliable_g = boolean(events['unreliable_g_photometry'])

# Fit-level CMD requirements inherited from the current absolute-CMD generation.
fit_quality = (
    np.isfinite(g0) & np.isfinite(g0_error)
    & np.isfinite(bp0) & np.isfinite(rp0)
    & np.isfinite(colour_error) & (colour_error < 1.0)
    & np.isfinite(source_parallax) & (source_parallax > 0)
    & np.isfinite(source_parallax_error) & (source_parallax_error > 0)
    & (source_parallax_snr > 2.0)
)

# New Gaia/all_source data-quality selection.
catalogue_quality = (
    ~unreliable_g
    & np.isfinite(ruwe) & (ruwe < MAX_RUWE)
    & np.isfinite(visibility_periods) & (visibility_periods >= MIN_VISIBILITY_PERIODS)
    & np.isfinite(g_flux_snr) & (g_flux_snr > MIN_G_FLUX_SNR)
    & np.isfinite(bp_flux_snr) & (bp_flux_snr > MIN_BP_RP_FLUX_SNR)
    & np.isfinite(rp_flux_snr) & (rp_flux_snr > MIN_BP_RP_FLUX_SNR)
)

valid = fit_quality & catalogue_quality
print(f'Fit-level CMD selection: {fit_quality.sum():,} / {len(events):,}')
print(f'Gaia/all_source quality selection: {catalogue_quality.sum():,} / {len(events):,}')
print(f'Final quality-cut source CMD: {valid.sum():,} / {len(events):,}')

## Automated Gaia DR4 reference background

The archive's `random_index` defines a random ordering. Query **all** qualifying
rows with `0 <= random_index < REFERENCE_RANDOM_INDEX_LIMIT`; no sky, brightness
or source-ID selection is made. The default window covers two million random
indices. No SQL `TOP` truncation is applied. An independent count and TAP status
checks verify the entire bounded result was returned.

The reference requires positive parallax/error and parallax S/N > 5, RUWE < 1.4,
at least nine visibility periods, G flux S/N > 50, BP/RP flux S/N > 20, reliable
G photometry, and finite CMD quantities. The configurable thresholds are below.
Dereddened plots additionally require finite DR4 `ag` and `ebpminrp` values.
After these cuts, select at most 150,000 stars without replacement using a fixed
seed. If fewer qualify, use all of them and report the actual number. Increase
the random-index window to obtain a larger pool.

This represents the **quality-selected catalogue subset**, not an unbiased
Galactic population: parallax precision, photometric quality and extinction
availability affect which stars can enter. No completeness correction is applied.

On first use, inspect current column metadata and run sequential asynchronous
queries using the same resumable job/overflow checks as notebook 02. Subsequent
runs reuse the checksummed ECSV sample without contacting the archive. Endpoint,
schema, release, query, cuts, seed and sample cap identify each cached sample.
Changing these settings creates a separate sample. Raw queries and job records
are retained under `data/dr4_cmd_reference/`.

In [ ]:
def now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2))

def source_id(value):
    if isinstance(value, (float, np.floating)):
        raise ValueError("A floating-point source ID may already have lost precision.")
    value = str(value)
    if not re.fullmatch(r"[0-9]{1,19}", value) or int(value) <= 0:
        raise ValueError("Source IDs must be positive decimal strings or exact integers.")
    return value

def identifier(value):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", value):
        raise ValueError("Invalid ADQL identifier")
    return value

def authenticate(gaia):
    if not GAIA_USERNAME.strip() or GAIA_USERNAME == "YOUR_GAIA_USERNAME":
        raise ValueError("Set GAIA_USERNAME to your own authorised archive account.")
    captured = None
    password = None
    try:
        if AUTH_METHOD == "keychain":
            captured = subprocess.run(
                ["security", "find-generic-password", "-a", GAIA_USERNAME,
                 "-s", KEYCHAIN_SERVICE, "-w"],
                capture_output=True, text=True, check=True, timeout=15)
            password = captured.stdout.rstrip("\n")
        elif AUTH_METHOD == "prompt":
            password = getpass("Gaia archive password: ")
        else:
            raise ValueError("AUTH_METHOD must be prompt or keychain")
        if not password:
            raise ValueError("Empty password")
        gaia.login(user=GAIA_USERNAME, password=password, verbose=False)
    finally:
        del password, captured
        # Astroquery also retains passwords internally; discard those references.
        for client in (gaia, getattr(gaia, "_GaiaClass__gaiadata", None)):
            if client is not None and hasattr(client, "_TapPlus__pwd"):
                client._TapPlus__pwd = None

def checked_votable(path):
    payload = path.read_bytes()
    if payload[:2] == b"\x1f\x8b":
        payload = gzip.decompress(payload)
    root = ET.fromstring(payload)
    statuses = [n.get("value", "").upper() for n in root.iter()
                if n.tag.split("}")[-1] == "INFO" and n.get("name", "").upper() == "QUERY_STATUS"]
    if not statuses or any(s != "OK" for s in statuses):
        raise RuntimeError(f"TAP output is incomplete or failed: QUERY_STATUS={statuses}")
    result = Table.read(BytesIO(payload), format="votable")
    # Normalize archive column names without changing values or units.
    for column in list(result.colnames):
        if column.lower() != column:
            result.rename_column(column, column.lower())
    return result

def tap(gaia, query, name, refresh=False):
    signature = json.dumps(dict(CONFIG, query=query), sort_keys=True)
    key = name + "_" + hashlib.sha256(signature.encode()).hexdigest()[:12]
    folder = OUTPUT / "tap" / key
    folder.mkdir(parents=True, exist_ok=True)
    result_file, ledger = folder / "result.vot", folder / "job.json"
    (folder / "query.adql").write_text(query)
    state = json.loads(ledger.read_text()) if ledger.exists() else dict(CONFIG, query=query, created=now())
    if state.get("status") == "complete":
        if result_file.exists() and not refresh:
            return checked_votable(result_file)
        if refresh:
            # Refresh only a completed query. A submitted/timed-out job is resumed.
            state = dict(CONFIG, query=query, created=now())
    if state.get("job_id"):
        job = gaia.load_async_job(jobid=state["job_id"], load_results=False, verbose=False)
    else:
        jobs = gaia.list_async_jobs(verbose=False) if AUTHENTICATE else []
        phases = [j.get_phase().upper() for j in jobs]
        if phases.count("EXECUTING") >= 5 or phases.count("QUEUED") >= 100:
            raise RuntimeError("Shared account job capacity is full; inspect existing jobs before retrying.")
        job = gaia.launch_job_async(query, name=name, output_file=str(result_file),
            output_format="votable", dump_to_file=True, background=True, verbose=False)
        state.update(job_id=str(job.jobid), status="submitted")
        save_json(ledger, state)  # Persist immediately, before polling.
    deadline = time.monotonic() + JOB_WAIT_SECONDS
    while True:
        phase = job.get_phase(update=True).upper()
        state.update(phase=phase, checked=now())
        save_json(ledger, state)
        if phase == "COMPLETED":
            break
        if phase in {"ERROR", "ABORTED"}:
            raise RuntimeError(f"TAP job {job.jobid} is {phase}; inspect it before replacement.")
        if time.monotonic() > deadline:
            raise TimeoutError(f"Job {job.jobid} retained for resumption; no replacement was submitted.")
        time.sleep(2)
    job.outputFile = str(result_file)
    job.outputFileUser = str(result_file)
    job.save_results(verbose=False)  # Preserve the original server response.
    result = checked_votable(result_file)
    result.write(folder / "result.ecsv", format="ascii.ecsv", overwrite=True)
    state.update(status="complete", rows=len(result), completed=now(), query_status_checked=True)
    save_json(ledger, state)
    return result


@contextmanager
def archive_session():
    identifier(TAP_SCHEMA)
    previous_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(NETWORK_TIMEOUT_SECONDS)
    gaia = None
    try:
        from astroquery.gaia import GaiaClass
        gaia = GaiaClass(gaia_tap_server=ARCHIVE_URL, gaia_data_server=ARCHIVE_URL,
                         show_server_messages=False)
        gaia.ROW_LIMIT = -1
        if AUTHENTICATE:
            authenticate(gaia)
        smoke = tap(gaia, f"SELECT source_id FROM {TAP_SCHEMA}.gaia_source "
                    f"WHERE source_id={source_id(SMOKE_SOURCE_ID)}", "smoke", refresh=True)
        if len(smoke) != 1:
            raise RuntimeError("Known-source TAP test failed; check endpoint, schema and access.")
        yield gaia
    finally:
        try:
            if gaia is not None and AUTHENTICATE:
                gaia.logout(verbose=False)
        finally:
            socket.setdefaulttimeout(previous_timeout)

def inspect_columns(gaia, table_names):
    records, columns = [], {}
    for name in table_names:
        identifier(name)
        table = gaia.load_table(f"{TAP_SCHEMA}.{name}", verbose=False)
        columns[name] = [str(c.name).lower() for c in table.columns]
        if "source_id" not in columns[name]:
            raise RuntimeError(f"Metadata does not advertise {name}.source_id.")
        records.append(dict(table=f"{TAP_SCHEMA}.{name}", columns=columns[name]))
    save_json(OUTPUT / "columns.json", dict(CONFIG, inspected=now(), tables=records))
    return columns




CONFIG = dict(endpoint=ARCHIVE_URL, schema=TAP_SCHEMA, data_release=DATA_RELEASE)
OUTPUT = REFERENCE_CACHE

def reference_request():
    identifier(TAP_SCHEMA)
    for name, value in [('REFERENCE_RANDOM_INDEX_LIMIT', REFERENCE_RANDOM_INDEX_LIMIT),
                        ('REFERENCE_MAX_STARS', REFERENCE_MAX_STARS)]:
        if isinstance(value, bool) or not isinstance(value, int) or not 0 < value <= np.iinfo(np.int64).max:
            raise ValueError(f'{name} must be a positive integer within signed 64-bit range.')
    if isinstance(REFERENCE_SEED, bool) or not isinstance(REFERENCE_SEED, int) or REFERENCE_SEED < 0:
        raise ValueError('REFERENCE_SEED must be a nonnegative integer.')
    if not isinstance(MIN_VISIBILITY_PERIODS, int) or MIN_VISIBILITY_PERIODS < 0:
        raise ValueError('MIN_VISIBILITY_PERIODS must be a nonnegative integer.')
    cuts = dict(min_parallax_snr=float(REFERENCE_MIN_PARALLAX_SNR), max_ruwe=float(MAX_RUWE),
                min_visibility_periods=MIN_VISIBILITY_PERIODS,
                min_g_flux_snr=float(MIN_G_FLUX_SNR), min_bp_rp_flux_snr=float(MIN_BP_RP_FLUX_SNR))
    if not all(np.isfinite(v) and v > 0 for k, v in cuts.items() if k != 'min_visibility_periods'):
        raise ValueError('Reference S/N and RUWE thresholds must be finite and positive.')
    columns = ['source_id', 'random_index', 'bp_rp', 'phot_g_mean_mag', 'parallax',
               'parallax_error', 'parallax_over_error', 'ruwe', 'visibility_periods_used',
               'phot_g_mean_flux_over_error', 'phot_bp_mean_flux_over_error',
               'phot_rp_mean_flux_over_error', 'unreliable_g_photometry']
    where = [f'random_index >= 0 AND random_index < {REFERENCE_RANDOM_INDEX_LIMIT}',
             'parallax > 0', 'parallax_error > 0',
             f"parallax_over_error > {cuts['min_parallax_snr']}",
             f"ruwe > 0 AND ruwe < {cuts['max_ruwe']}",
             f"visibility_periods_used >= {cuts['min_visibility_periods']}",
             f"phot_g_mean_flux_over_error > {cuts['min_g_flux_snr']}",
             f"phot_bp_mean_flux_over_error > {cuts['min_bp_rp_flux_snr']}",
             f"phot_rp_mean_flux_over_error > {cuts['min_bp_rp_flux_snr']}",
             "unreliable_g_photometry = 'false'", 'bp_rp IS NOT NULL',
             'phot_g_mean_mag IS NOT NULL']
    if DEREDDEN:
        columns += ['ag', 'ebpminrp']
        where += ['ag IS NOT NULL', 'ebpminrp IS NOT NULL']
    selection = '\n  AND '.join(where)
    table = f'{TAP_SCHEMA}.gaia_source'
    query = f"SELECT {', '.join(columns)}\nFROM {table}\nWHERE {selection}"
    count_query = f'SELECT COUNT(*) AS n FROM {table}\nWHERE {selection}'
    request = dict(CONFIG, query=query, count_query=count_query, cuts=cuts,
                   random_index_limit=REFERENCE_RANDOM_INDEX_LIMIT, max_stars=REFERENCE_MAX_STARS,
                   seed=REFERENCE_SEED, deredden=bool(DEREDDEN))
    return request, columns

def checked_reference_rows(table, request, columns):
    if missing := set(columns) - set(table.colnames):
        raise RuntimeError(f'Reference lacks required columns: {sorted(missing)}')
    ids = [source_id(value) for value in table['source_id']]
    if len(set(ids)) != len(ids):
        raise RuntimeError('Reference contains duplicate source IDs.')
    index = np.ma.asarray(table['random_index'])
    if index.dtype.kind not in 'iu' or np.any(np.ma.getmaskarray(index)):
        raise RuntimeError('random_index must contain exact, unmasked integers.')
    if np.any(index < 0) or np.any(index >= request['random_index_limit']):
        raise RuntimeError('Reference includes rows outside the requested random-index window.')
    if len(np.unique(index)) != len(index):
        raise RuntimeError('Reference random indices are not unique.')
    values = {name: numeric(table[name]) for name in columns
              if name not in ('source_id', 'random_index', 'unreliable_g_photometry')}
    keep = np.ones(len(table), dtype=bool)
    for value in values.values():
        keep &= np.isfinite(value)
    cuts = request['cuts']
    keep &= ((values['parallax'] > 0) & (values['parallax_error'] > 0)
             & (values['parallax_over_error'] > cuts['min_parallax_snr'])
             & (values['ruwe'] > 0) & (values['ruwe'] < cuts['max_ruwe'])
             & (values['visibility_periods_used'] >= cuts['min_visibility_periods'])
             & (values['phot_g_mean_flux_over_error'] > cuts['min_g_flux_snr'])
             & (values['phot_bp_mean_flux_over_error'] > cuts['min_bp_rp_flux_snr'])
             & (values['phot_rp_mean_flux_over_error'] > cuts['min_bp_rp_flux_snr'])
             & ~boolean(table['unreliable_g_photometry'])
             & ~np.ma.getmaskarray(table['unreliable_g_photometry']))
    return table[keep].copy()

def load_dr4_reference():
    request, columns = reference_request()
    key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()[:16]
    folder = REFERENCE_CACHE / 'samples' / key
    sample_file, manifest_file = folder / 'sample.ecsv', folder / 'provenance.json'
    if sample_file.exists() and manifest_file.exists():
        info = json.loads(manifest_file.read_text())
        if info.get('request') != request or info.get('sha256') != hashlib.sha256(sample_file.read_bytes()).hexdigest():
            raise RuntimeError(f'Cached reference provenance/checksum failed: {folder}; inspect it before reuse.')
        sample = Table.read(sample_file, format='ascii.ecsv')
        checked = checked_reference_rows(sample, request, columns)
        if len(checked) != len(sample) or not 0 < len(sample) <= request['max_stars'] or len(sample) != info['sample_rows']:
            raise RuntimeError('Cached reference violates its recorded selection/count.')
        print(f'Cached Gaia DR4 CMD reference: {len(sample):,} stars from {sample_file}')
        return sample, info

    REFERENCE_CACHE.mkdir(parents=True, exist_ok=True)
    with archive_session() as gaia:
        metadata = inspect_columns(gaia, ['gaia_source'])
        if missing := set(columns) - set(metadata['gaia_source']):
            raise RuntimeError(f'Current DR4 metadata lacks required columns: {sorted(missing)}')
        counts = tap(gaia, request['count_query'], 'cmd_reference_count')
        raw = tap(gaia, request['query'], 'cmd_reference_random_window')
    if len(counts) != 1 or len(raw) != int(counts['n'][0]):
        raise RuntimeError('Reference row count disagrees with COUNT(*); partial results cannot be used.')
    eligible = checked_reference_rows(raw, request, columns)
    if not len(eligible):
        raise RuntimeError('No finite DR4 stars pass the reference cuts; increase the random-index window or review the cuts.')
    eligible.sort('random_index')  # Stable seed behavior regardless of server row order.
    count = min(request['max_stars'], len(eligible))
    selected = np.sort(np.random.default_rng(request['seed']).choice(len(eligible), size=count, replace=False))
    sample = eligible[selected]
    sample.meta.update(data_release=DATA_RELEASE, tap_schema=TAP_SCHEMA,
                       sampling='bounded random_index window, quality cuts, seeded cap without replacement')
    folder.mkdir(parents=True, exist_ok=True)
    partial = folder / 'sample.part.ecsv'
    try:
        sample.write(partial, format='ascii.ecsv', overwrite=True)
        partial.replace(sample_file)
    finally:
        partial.unlink(missing_ok=True)
    info = dict(request=request, created_utc=now(), sample_file=str(sample_file),
                raw_rows=len(raw), finite_quality_rows=len(eligible), sample_rows=len(sample),
                sha256=hashlib.sha256(sample_file.read_bytes()).hexdigest(),
                tap_cache_directory=str(OUTPUT / 'tap'), numpy_version=np.__version__,
                selection_note='Random all-sky subset of quality-eligible DR4 sources; not completeness-corrected.')
    save_json(manifest_file, info)
    print(f'Downloaded {len(raw):,} rows; {len(eligible):,} pass finite/quality cuts; using {len(sample):,} reference stars.')
    print(f'Cached sample and provenance in {folder}')
    return sample, info

In [ ]:
absolute_g = np.full(len(events), np.nan)
absolute_g[valid] = g0[valid] + 5 * np.log10(source_parallax[valid]) - 10
absolute_g_error = np.full(len(events), np.nan)
absolute_g_error[valid] = np.hypot(
    g0_error[valid],
    (5 / np.log(10)) * source_parallax_error[valid] / source_parallax[valid],
)
plot_colour = colour.copy()

if DEREDDEN:
    ag = numeric(events['ag'])
    ebpminrp = numeric(events['ebpminrp'])
    extinction_quality = np.isfinite(ag) & np.isfinite(ebpminrp)
    valid &= extinction_quality
    plot_colour -= ebpminrp
    absolute_g[valid] -= ag[valid]
    print(f'After finite extinction corrections: {valid.sum():,} sources')

if USE_REFERENCE_BACKGROUND:
    reference, reference_provenance = load_dr4_reference()
    reference_colour = numeric(reference['bp_rp'])
    reference_g = numeric(reference['phot_g_mean_mag'])
    reference_parallax = numeric(reference['parallax'])
    reference_valid = np.isfinite(reference_colour) & np.isfinite(reference_g) & (reference_parallax > 0)
    reference_colour = reference_colour[reference_valid]
    reference_absolute_g = reference_g[reference_valid] + 5 * np.log10(reference_parallax[reference_valid]) - 10
    if DEREDDEN:
        reference_colour -= numeric(reference['ebpminrp'])[reference_valid]
        reference_absolute_g -= numeric(reference['ag'])[reference_valid]

In [ ]:
x_range, y_range = (-0.6, 2.7), (-5.0, 10.5)
fig, ax = plt.subplots(figsize=(6.2, 6.2), constrained_layout=True)
if USE_REFERENCE_BACKGROUND:
    density, x_edges, y_edges = np.histogram2d(
        reference_colour, reference_absolute_g, bins=(260, 300), range=(x_range, y_range)
    )
    ax.pcolormesh(
        x_edges, y_edges, density.T, cmap='Greys', alpha=0.62, shading='auto',
        norm=LogNorm(vmin=1, vmax=max(2, density.max())), rasterized=True,
    )
    ax.text(0.02, 0.02, f'Gaia DR4 background: {len(reference):,} stars',
            transform=ax.transAxes, fontsize=8, va='bottom',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
ax.errorbar(
    plot_colour[valid], absolute_g[valid], xerr=colour_error[valid], yerr=absolute_g_error[valid],
    fmt='none', ecolor='#1769aa', elinewidth=0.8, alpha=0.36, capsize=0, rasterized=True, zorder=2,
)
ax.scatter(
    plot_colour[valid], absolute_g[valid], s=22, color='#1769aa', alpha=0.75,
    edgecolors='white', linewidths=0.3, rasterized=True, zorder=3,
)
ax.invert_yaxis()
ax.set(xlim=x_range, ylim=(y_range[1], y_range[0]))
ax.grid(True, color='0.88', linewidth=0.7)
ax.set_axisbelow(True)
if DEREDDEN:
    ax.set_xlabel(r'Dereddened baseline source colour $(BP_0-RP_0)-E(BP-RP)$ [mag]')
    ax.set_ylabel(r'Extinction-corrected absolute source magnitude $M_{G,0}$ [mag]')
else:
    ax.set_xlabel(r'Baseline source colour $BP_0-RP_0$ [mag]')
    ax.set_ylabel(r'Absolute source magnitude $M_{G,0}$ [mag]')
ax.set_title(f'Quality-cut microlensing sources ({valid.sum():,} events)')

suffix = '_dereddened' if DEREDDEN else ''
figure_png = OUTPUT_STEM.with_name(OUTPUT_STEM.name + suffix + '.png')
figure_pdf = OUTPUT_STEM.with_name(OUTPUT_STEM.name + suffix + '.pdf')
fig.savefig(figure_png, dpi=220)
fig.savefig(figure_pdf)
plt.show()
print(figure_png)
print(figure_pdf)

In [ ]:
output = Table()
output['source_id'] = events['source_id'][valid]
output['paczynski0_g0'] = g0[valid]
output['paczynski0_g0_error'] = g0_error[valid]
output['paczynski0_bp_rp'] = colour[valid]
output['paczynski0_bp_rp_error'] = colour_error[valid]
output['paczynski3_varpi_source_mas'] = source_parallax[valid]
output['paczynski3_varpi_source_snr'] = source_parallax_snr[valid]
output['absolute_g'] = absolute_g[valid]
output['absolute_g_error'] = absolute_g_error[valid]
output['ruwe'] = ruwe[valid]
output['visibility_periods_used'] = visibility_periods[valid]
output['phot_g_mean_flux_over_error'] = g_flux_snr[valid]
output['phot_bp_mean_flux_over_error'] = bp_flux_snr[valid]
output['phot_rp_mean_flux_over_error'] = rp_flux_snr[valid]
if DEREDDEN:
    output['ag'] = ag[valid]
    output['ebpminrp'] = ebpminrp[valid]
    output['dereddened_bp_rp'] = plot_colour[valid]

data_path = OUTPUT_STEM.with_name(OUTPUT_STEM.name + ('_dereddened' if DEREDDEN else '') + '_data.csv')
output.write(data_path, format='ascii.csv', overwrite=True)
print(f'Saved {len(output):,} selected sources to {data_path}')
output[:10]

plot_provenance = {
    'created_utc': now(), 'deredden': bool(DEREDDEN),
    'microlensing_input': str(INPUT_PATH),
    'microlensing_input_sha256': hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest(),
    'selected_microlensing_sources': int(valid.sum()),
    'reference': reference_provenance if USE_REFERENCE_BACKGROUND else None,
    'output_files': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                     for p in (figure_png, figure_pdf, data_path)},
}
provenance_path = OUTPUT_STEM.with_name(OUTPUT_STEM.name + ('_dereddened' if DEREDDEN else '') + '_provenance.json')
save_json(provenance_path, plot_provenance)
print(f'Plot provenance: {provenance_path}')